# CMIP6 vs. AMIP (coarse-resolution HighResMIP) JAS Precipitation Bias

Two-panel figure comparing multi-model-ensemble-mean JAS (Jul-Aug-Sep) precipitation bias vs.
IMERG observations:

- **Panel (a) "CMIP"**: full CMIP6 ensemble (49 models), from `cmip6.pr.climo.nc`.
- **Panel (b) "AMIP"**: coarse-resolution ("AMIP-style", `highresSST-present`) subset of the
  HighResMIP ensemble (6 models), from `HighResMIP.cmip6.pr.climo.nc`.

Modeled after the ensemble-mean-bias approach in `HighResMIP_prec_historical_biases.ipynb`, but
this notebook runs **locally** against OneDrive-synced data rather than on NASA Discover, and the
source climatology files use a `month` dimension (pre-computed monthly climatology) rather than a
`time` timeseries dimension, so seasonal means are computed directly with `.sel(month=...)` rather
than `data_funcs.jas_seasonal_mean()`.


In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import xarray as xr
xr.set_options(keep_attrs=True)
import numpy as np

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import cm

%config InlineBackend.figure_format = 'retina'

# add path to custom functions (repo convention)
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path + "/py_functions")
from colorbar_funcs import *
from data_funcs import *
from stats_funcs import *
from domain_funcs import *

# Note: no Discover cartopy.config[...] override needed here -- this notebook runs locally,
# and the local cartopy install already has Natural Earth shapefiles cached
# (~/.local/share/cartopy).


## Paths and ensemble membership

In [ ]:
dpath_cmip6 = '/Users/dervlamk/OneDrive/data/cmip6/cmip6.pr.climo.nc'
dpath_hrmip = '/Users/dervlamk/OneDrive/data/highresmip/HighResMIP.cmip6.pr.climo.nc'
dpath_obs   = '/Users/dervlamk/OneDrive/data/obs/satellite/imerg/imerg.climo.gn.nc'
dpath_figs  = '../figs/'

# Coarse-resolution ("AMIP"-style) HighResMIP subset for panel (b).
# EC-Earth3P is deliberately excluded here -- it groups with the fine-resolution/HR set
# (consistent with HighResMIP_prec_historical_biases.ipynb's hr_map, where 'EC-Earth3P' is
# listed alongside 'EC-Earth3P-HR', not in the LR set).
amip_cfgs = ['IPSL-CM6A-LR', 'FGOALS-f3-L', 'CAMS-CSM1-0',
             'ECMWF-IFS-LR', 'HadGEM3-GC31-LM', 'CNRM-CM6-1']


## IMERG observations: JAS climatology

**Units note**: `imerg.climo.gn.nc` carries no `units` attribute. The reference notebook's raw
IMERG *timeseries* files are in mm/hr and need `*24`, but a raw-value check here (global mean
~3.0, max ~69, JAS-Amazon-box mean ~2.8) shows this precomputed climatology is **already in
mm/day** -- applying `*24` would inflate it ~24x into an implausible range. So no conversion is
applied, only a units label.


In [ ]:
obs = xr.open_dataset(dpath_obs).precipitation
obs.attrs['units'] = 'mm/day'   # already mm/day -- see units note above, verified via raw-value check

# plain (unweighted) mean of the 3 climatological JAS months -- jas_seasonal_mean() doesn't apply
# here since it expects a 'time.month' timeseries accessor, not a pre-computed 'month' dim.
obs_jas = obs.sel(month=[7, 8, 9]).mean(dim='month')

print('IMERG JAS mean over Amazon box (5S-0, 65W-55W):',
      float(obs_jas.sel(lat=slice(-10, 0), lon=slice(-65, -55)).mean()), 'mm/day')


## Panel (a): CMIP6 ensemble mean bias

`cmip6.pr.climo.nc` already provides all 49 models on one common 2°x2.5° grid (lon 0:360), so no
per-model regridding is needed -- only IMERG needs to be interpolated onto that shared grid.

**Units note**: `pr` here has no `units` attribute either. CMIP6 `pr` is normally a flux
(`kg m-2 s-1`, needing `*86400`), but a raw-value check shows this file's per-model JAS means are
already O(2-3) with a max of ~73 -- i.e. **already mm/day**, not a flux. Applying `*86400` would
inflate the ensemble mean to ~2x10^5 mm/day, which is obviously unphysical. No conversion applied.


In [ ]:
cmip6 = xr.open_dataset(dpath_cmip6).pr
cmip6 = cmip6.squeeze(drop=True)   # drop singleton member_id, dcpp_init_year
cmip6.attrs['units'] = 'mm/day'    # already mm/day -- see units note above, verified via raw-value check
print('CMIP6 dims after squeeze:', cmip6.dims, dict(cmip6.sizes))

cmip6_jas = cmip6.sel(month=[7, 8, 9]).mean(dim='month')   # (source_id, lat, lon)
print('CMIP6 JAS mean over Amazon box, first 3 models:',
      cmip6_jas.sel(lat=slice(-10, 0), lon=slice(295, 305))
               .mean(dim=['lat', 'lon']).isel(source_id=slice(0, 3)).values)

cmip6_mme = cmip6_jas.mean(dim='source_id')   # (lat, lon), direct mean on the common grid

obs_on_cmip6 = lonFlip(obs_jas).interp(lat=cmip6_mme.lat, lon=cmip6_mme.lon)
bias_cmip = cmip6_mme - obs_on_cmip6
bias_cmip.attrs['units'] = 'mm/day'

print('n source_id in CMIP6 ensemble:', cmip6.sizes['source_id'])
print('CMIP bias min/mean/max:', float(bias_cmip.min()), float(bias_cmip.mean()), float(bias_cmip.max()))


## Panel (b): AMIP (coarse-resolution HighResMIP) ensemble mean bias

`HighResMIP.cmip6.pr.climo.nc` declares all 23 model configs merged onto one sparse union
lat/lon axis (`lat=7777, lon=7514`, nominal size 3 TB) -- for any given `source_id`, only the
index positions matching that model's own grid should be populated; everything else NaN.
`member_id` (22) carries no coordinate labels, so for each config we'd normally diagnose which
`member_id` index holds data, then strip the NaN padding to recover that model's native grid.

> **⚠️ Blocked: this file currently contains no usable `pr` data.** The file is only 141 KB on
> disk (vs. the 3 TB the declared dimensions imply), and direct checks find **zero non-null
> values anywhere in `pr`** for every model tested (`IPSL-CM6A-LR`, and a `month=0` scan across
> all 23 `source_id` values) -- not just in the 6 `amip_cfgs` subset. This looks like an
> incomplete/failed export (coordinates and metadata were written, but the actual data array
> was not), not a bug in this notebook's extraction logic. A full-array `notnull().sum()` over
> the whole variable also does not complete in a reasonable time, consistent with an
> HDF5/NetCDF4 reader having to scan a mostly-nonexistent 3 TB declared range.
>
> The cells below implement the intended per-model extraction (member_id diagnosis + NaN-strip +
> programmatic coarsest-grid selection) and should work once a valid `HighResMIP.cmip6.pr.climo.nc`
> is available, but **will currently raise an `AssertionError`** on the first config (no populated
> `member_id` found) because there is no real data to find. Confirm with the user whether the file
> needs to be regenerated, or whether a different path/variable holds the actual HighResMIP data,
> before relying on this section.


In [ ]:
hrmip = xr.open_dataset(dpath_hrmip, chunks={'source_id': 1, 'month': 12, 'member_id': -1,
                                              'lat': 500, 'lon': 500}).pr

def extract_model_native(ds_pr, cfg):
    """Isolate one source_id's true native grid + populated member_id from the sparse merged file."""
    sub = ds_pr.sel(source_id=cfg)   # (month, member_id, dcpp_init_year, lat, lon)
    counts = sub.isel(dcpp_init_year=0).notnull().sum(dim=['lat', 'lon', 'month']).compute()
    populated = np.nonzero(counts.values)[0]
    print(f'{cfg}: populated member_id index/indices = {populated.tolist()} '
          f'(counts: {counts.values[populated].tolist()})')
    assert len(populated) >= 1, f'{cfg}: no populated member_id found'
    if len(populated) > 1:
        print(f'  WARNING: {cfg} has multiple populated member_id slots; using the first ({populated[0]}).')
    sub = sub.isel(member_id=int(populated[0]), dcpp_init_year=0)
    sub = sub.dropna(dim='lat', how='all').dropna(dim='lon', how='all')
    return sub.load()   # small once stripped down to native grid size -- safe to load now


In [ ]:
amip_jas = {}
grid_sizes = {}
for cfg in amip_cfgs:
    print(f'Extracting {cfg}...')
    da = extract_model_native(hrmip, cfg) * 86400   # kg m-2 s-1 -> mm/day
    da.attrs['units'] = 'mm/day'
    amip_jas[cfg] = da.sel(month=[7, 8, 9]).mean(dim='month')
    grid_sizes[cfg] = da.sizes['lat'] * da.sizes['lon']
    print(f'  native grid: {da.sizes["lat"]} x {da.sizes["lon"]} = {grid_sizes[cfg]} points; '
          f'lon range [{float(da.lon.min())}, {float(da.lon.max())}]')

coarsest_cfg = min(grid_sizes, key=grid_sizes.get)
print('Coarsest of the 6 AMIP configs (regrid target):', coarsest_cfg, grid_sizes[coarsest_cfg])


In [ ]:
target = amip_jas[coarsest_cfg]

amip_bias_native = {}
for cfg, da in amip_jas.items():
    obs_regrid = lonFlip(obs_jas).interp(lat=da.lat, lon=da.lon)
    amip_bias_native[cfg] = da - obs_regrid

slices = []
for cfg, bias in amip_bias_native.items():
    tagged = bias.assign_coords(source_id=cfg)
    slices.append(tagged.interp(lat=target.lat.values, lon=target.lon.values))
amip_mme = xr.concat(slices, dim='source_id').mean(dim='source_id')
amip_mme.attrs['units'] = 'mm/day'

print('n source_id in AMIP ensemble:', len(amip_cfgs))
print('AMIP MME bias min/mean/max:', float(amip_mme.min()), float(amip_mme.mean()), float(amip_mme.max()))


### Sanity check: one model's native-grid bias before averaging

Visually confirm the bias field looks physically sane (dry/wet bands in plausible locations, no
grid artifacts from the `dropna` step) before trusting the ensemble mean.


In [ ]:
dcmap, _, _, _ = get_settings(field='precip', diff=True)
_check_levels = np.linspace(-6, 6, 25)
_check_norm = mpl.colors.BoundaryNorm(_check_levels, dcmap.N)

fig, ax = plt.subplots(figsize=(7, 4), subplot_kw={'projection': ccrs.PlateCarree()})
check_field = amip_bias_native[coarsest_cfg]
cf = ax.pcolormesh(check_field.lon, check_field.lat, check_field,
                    cmap=dcmap, norm=_check_norm, transform=ccrs.PlateCarree())
ax.coastlines(linewidth=1)
ax.set_global()
ax.set_title(f'{coarsest_cfg} native-grid bias (sanity check)')
fig.colorbar(cf, ax=ax, orientation='horizontal', shrink=0.7, extend='both')
plt.show()


## Final figure: CMIP vs. AMIP ensemble-mean JAS precipitation bias

In [ ]:
dcmap, _, _, _ = get_settings(field='precip', diff=True)   # combine_cmaps(BrBG, Blues)
dlevels = np.linspace(-6, 6, 25)                            # get_settings' own diff bounds (global map)
dnorm = mpl.colors.BoundaryNorm(dlevels, dcmap.N)

fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                          subplot_kw={'projection': ccrs.PlateCarree()},
                          constrained_layout=True)

for ax, field, title in zip(axes, [bias_cmip, amip_mme], ['CMIP', 'AMIP']):
    cf = ax.pcolormesh(field.lon, field.lat, field, cmap=dcmap, norm=dnorm,
                        transform=ccrs.PlateCarree())
    ax.coastlines(linewidth=1)
    ax.add_feature(cfeature.BORDERS, edgecolor='k', linewidth=0.5)
    ax.set_global()
    gl = ax.gridlines(crs=ccrs.PlateCarree(), lw=.5, color='gray', linestyle='--', draw_labels=True)
    gl.top_labels = False
    gl.right_labels = False
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    ax.set_title(title, fontsize=14, fontweight='bold')

cbar = fig.colorbar(cf, ax=axes, orientation='horizontal', shrink=0.5, pad=0.08, extend='both')
cbar.set_label('JAS Precipitation Bias vs. IMERG [mm/day]', fontsize=11)

fig.savefig(dpath_figs + 'cmip6_amip_jas_prec_bias_mme.pdf', bbox_inches='tight')
fig.savefig(dpath_figs + 'cmip6_amip_jas_prec_bias_mme.png', bbox_inches='tight', dpi=200)
plt.show()


## Verification checklist

1. Run top-to-bottom in the `climate` conda environment/kernel.
2. Confirm CMIP6 ensemble count prints `49`; confirm each AMIP config's `member_id` diagnostic
   finds exactly one populated index (investigate before trusting a model if more than one shows up).
3. Confirm unit-converted JAS values are physically plausible (e.g. Amazon-box mean ~4-10 mm/day),
   not implausible (near-zero or >100) -- this validates the `*86400`/`*24` unit assumptions since
   neither source file has a units attribute.
4. Inspect the printed native grid sizes for the 6 AMIP configs and the chosen `coarsest_cfg`.
5. Inspect the single-model sanity bias plot for grid artifacts/seams before trusting `amip_mme`.
6. Open the final saved PNG/PDF and confirm both panels are titled "CMIP" and "AMIP", the colorbar
   range isn't badly saturated, and there are no dateline/pole seams in the AMIP panel.
